## 4 Generation of the Final Survey Sample

**Author:** Kristina Aleksandrovna Pedersen

This code samples political and entertainment videos to create the final video bundle for the respondents to be exposed to. We balance the political sample on strategy and party.

In [1]:
import pandas as pd
from glob import glob
import random
import os
import shutil
import re
import numpy as np

### Political sample

Note: this is the original sampling code, not taking into account the updated categories for the 11 videos where syncing issues occured during sample generation.

In [2]:
#Loading manually coded political data
political = pd.read_pickle("political.pkl").rename(columns = {'category': 'strategy'})
print(political.shape)
political.head(4)

(1228, 4)


,tiktok_id,party,user_name,strategy
0,7429725237949795616,Linke,die.linke.hamburg,neutral
1,7429402389330939168,Linke,die.linke.hamburg,neutral
2,7451969045944978710,AfD,peterfelser_mdb,neutral
3,7441489832104185110,AfD,eugen_schmidt.mdb,neutral


In [3]:
strategies = political.strategy.unique().tolist()
strategies

['neutral', 'anger', 'superiority', 'fear']

In [4]:
#Desired sample size for political videos = 200, num parties = 7. Include round up here
party_sample_size = 37 #(approx sample size given 200/7 videos per party + buffer)
print("Party sample size:", party_sample_size, "Full sample size:", party_sample_size *7)

#ideal sample size for each emotional category
ideal_sample_size = int(party_sample_size/4)
print(ideal_sample_size)

Party sample size: 37 Full sample size: 259
9


In [5]:
output = pd.DataFrame(columns = ['short_id', 'tiktok_id', 'party', 'user_name', 'video_desc', 
                                 'content', 'strategy', 'org_path', 'image_path'])
output

,short_id,tiktok_id,party,user_name,video_desc,content,strategy,org_path,image_path


In [6]:
#Iterate over each party
for party in political.party.unique():
    #temporary party dataframe
    partydf = political[political.party == party]

    #iterate over each emotional strategy
    for strategy in strategies:
        #a party and strategy specific dataframe
        catdf = partydf[partydf.strategy == strategy]
        #get number of observations
        num_videos = catdf.shape[0]

        #If number of obesrvations is above the ideal category sample size
        if num_videos >= ideal_sample_size:
            #then sample and save the sample to output dataframe
            cat_sample = catdf.sample(ideal_sample_size, random_state = 123)
            output = pd.concat([output, cat_sample])
        else:
            #if there are fewer observations then save the full dataframe to the output
            cat_sample = catdf
            output = pd.concat([output, cat_sample])

#Add political content type and user id column
output.content = 'political'
output.tiktok_id = output.tiktok_id.astype(str)
output.reset_index(drop = True, inplace = True)
print(output.shape)
output.head(3)

(219, 9)


,short_id,tiktok_id,party,user_name,video_desc,content,strategy,org_path,image_path
0,NaN,7457944217550097686,Linke,caren.lay.mdb,NaN,political,neutral,NaN,NaN
1,NaN,7461272586278702358,Linke,die.linke,NaN,political,neutral,NaN,NaN
2,NaN,7459710828896341280,Linke,die.linke.hamburg,NaN,political,neutral,NaN,NaN


In [7]:
output.strategy.value_counts()

strategy
neutral        63
superiority    56
anger          55
fear           45
Name: count, dtype: int64

In [8]:
#Include column with original video folder path
output.org_path = output.apply(lambda x: f"../videos/{x['party']}/@{x['user_name']}/{x['tiktok_id']}.mp4", axis = 1)
output.head(3)

,short_id,tiktok_id,party,user_name,video_desc,content,strategy,org_path,image_path
0,NaN,7457944217550097686,Linke,caren.lay.mdb,NaN,political,neutral,../videos/Linke/@caren.lay.mdb/745794421755009...,NaN
1,NaN,7461272586278702358,Linke,die.linke,NaN,political,neutral,../videos/Linke/@die.linke/7461272586278702358...,NaN
2,NaN,7459710828896341280,Linke,die.linke.hamburg,NaN,political,neutral,../videos/Linke/@die.linke.hamburg/74597108288...,NaN


In [9]:
#Add video description from metadata for simulator
for ix in output.index:
    video = output.tiktok_id[ix]
    metadata = pd.read_pickle(f"../video_metadata_clean/{output.party[ix]}/@{output.user_name[ix]}.pkl")
    output.loc[ix, 'video_desc'] = metadata[metadata.video_id == str(video)].video_desc.values[0]

output.head(3)

,short_id,tiktok_id,party,user_name,video_desc,content,strategy,org_path,image_path
0,NaN,7457944217550097686,Linke,caren.lay.mdb,Warum ich Videos auf TikTok mache und was ich ...,political,neutral,../videos/Linke/@caren.lay.mdb/745794421755009...,NaN
1,NaN,7461272586278702358,Linke,die.linke,Antwort auf @neequaye_ Am 23.2 alle Die Linke ...,political,neutral,../videos/Linke/@die.linke/7461272586278702358...,NaN
2,NaN,7459710828896341280,Linke,die.linke.hamburg,Wir sind Die Linke und unser Hamburg sieht so ...,political,neutral,../videos/Linke/@die.linke.hamburg/74597108288...,NaN


## Entertainment sample

In [ ]:
#all manually coded entertainment videos (multiple files coded across multiple RAs)
files2 = [file for file in glob("../samples/*") if "entertainment" in file]
df = pd.concat([pd.read_excel(file) for file in files2], ignore_index = True)
df.video_id = df.video_id.astype(str)
print(df.shape)
#remove the videos coded as political
df.dropna(subset = 'political', inplace = True)
print(df.shape)

(300, 10)
(295, 10)


In [12]:
#videos that were deleted
missing_vids = ['7447927352417357078',
 '7441326001117875474',
 '7448672982123236615',
 '7423189161168899348']
df = df[~df.video_id.isin(missing_vids)].reset_index(drop = True)
print(df.shape)

(294, 10)


In [13]:
#clean coded responses
df.political = df.political.replace('?', 1)
df.inappropriate_content = df.inappropriate_content.replace('?', 1)
df.inappropriate_content = df.inappropriate_content.fillna(0)
df.foreign_language = df.foreign_language.fillna(0)
df['notes'] = np.where(df.Notes.notnull(), 1, 0) #notes would mean that coder identified something off

/tmp/ipykernel_4529/3085627848.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.political = df.political.replace('?', 1)
/tmp/ipykernel_4529/3085627848.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.inappropriate_content = df.inappropriate_content.replace('?', 1)


In [14]:
#remove videos marked as political, not german or english, or inappropriate 
remove = df[['political',  'foreign_language', 'inappropriate_content', 'notes']].sum(axis = 1)
df = df[remove == 0].reset_index(drop = True)[['user_name', 'video_id']].rename(columns = {'video_id': 'tiktok_id'})
print(df.shape)
df.head(3)

(182, 2)


,user_name,tiktok_id
0,orangejuice_080,7435267582707518738
1,the_mannii,7445169087921130795
2,looooooooch,7450751578153192711


In [15]:
#Because this reduced the sample, we basically take that entire coded sample as our entertainment videos for the survey
ent_output = pd.DataFrame(columns = ['short_id', 'tiktok_id', 'party', 'user_name',
                                 'content', 'strategy', 'org_path', 'image_path'])
ent_output[['user_name', 'tiktok_id']] = df.sample(181, random_state = 123)
ent_output.tiktok_id = ent_output.tiktok_id.astype(str) 
ent_output['content'] = 'entertainment'
ent_output.org_path = ent_output.apply(lambda x: f"../entertainment/videos/{x['tiktok_id']}.mp4", axis = 1)
print(ent_output.shape)

#add video descriptions
metadata = pd.read_pickle("../entertainment/ent_video_metadata_clean.pkl")
metadata.video_id = metadata.video_id.astype(str)
metadata = metadata[['video_id', 'video_desc']].rename(columns = {'video_id': 'tiktok_id'})
metadata.drop_duplicates(subset = 'tiktok_id', keep = 'last', inplace = True)

(181, 8)


In [17]:
#merge ID information
ent_output = ent_output.merge(metadata, on = 'tiktok_id', how = 'left')
print(ent_output.shape)
ent_output.head(3)

(181, 9)


,short_id,tiktok_id,party,user_name,content,strategy,org_path,image_path,video_desc
0,NaN,7445076171861019947,NaN,ihearttherock333,entertainment,NaN,../entertainment/videos/7445076171861019947.mp4,NaN,
1,NaN,7451009742132006166,NaN,jean__kadyma099,entertainment,NaN,../entertainment/videos/7451009742132006166.mp4,NaN,#CapCut#wallpaper#🔥 #4k
2,NaN,7421909585490988294,NaN,cateblanka,entertainment,NaN,../entertainment/videos/7421909585490988294.mp4,NaN,#cate.blanka #foryoupage #newchallenge #lipsyn...


In [18]:
#concatinate the political and entertainment output for the final survey sample
final_output = pd.concat([output, ent_output], ignore_index = True)
newid = list(range(1, 400+1))
random.shuffle(newid)
final_output.short_id = newid
print(final_output.shape)
final_output.head(3)

(400, 9)


,short_id,tiktok_id,party,user_name,video_desc,content,strategy,org_path,image_path
0,8,7457944217550097686,Linke,caren.lay.mdb,Warum ich Videos auf TikTok mache und was ich ...,political,neutral,../videos/Linke/@caren.lay.mdb/745794421755009...,NaN
1,129,7461272586278702358,Linke,die.linke,Antwort auf @neequaye_ Am 23.2 alle Die Linke ...,political,neutral,../videos/Linke/@die.linke/7461272586278702358...,NaN
2,266,7459710828896341280,Linke,die.linke.hamburg,Wir sind Die Linke und unser Hamburg sieht so ...,political,neutral,../videos/Linke/@die.linke.hamburg/74597108288...,NaN


*Afterwards a path for the user profile image was added as well as a new path for the folder where the copies of the videos to were to be stored*